# Final Training — Load Best Params from Optuna

Загружает best_params из Optuna study (сохранено в rnn_otune.ipynb),
затем делает финальное обучение на 100 эпохах с этими параметрами.
Экспортирует ONNX модель и генерирует RNN_Scaler.mqh для EA.


In [1]:
import os, glob, io, json, sys, math
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import skl2onnx
from skl2onnx.common.data_types import FloatTensorType
import onnx

Device: cuda
Output dir: /content/drive/MyDrive/RNN/output
Artifacts dir: /content/drive/MyDrive/RNN/data_normalized


## 1. Load Best Params from Optuna

In [2]:
# Р—Р°РіСЂСѓР¶Р°РµРј best_params.json СЃРѕС…СЂР°РЅРµРЅРЅС‹Р№ РІ rnn_learn_v3_improved.ipynb
best_params_file = OUTPUT_DIR / 'best_params.json'

if not best_params_file.exists():
    raise FileNotFoundError(f'best_params.json not found: {best_params_file}')

with open(best_params_file, 'r') as f:
    optuna_results = json.load(f)

buy_params = optuna_results['best_params']
buy_metrics_optuna = optuna_results['metrics']
best_trial_num = optuna_results['best_trial_number']

print(f"Loaded Optuna results from Trial {best_trial_num}:")
print(f"  Precision: {optuna_results['best_value']:.4f}")
print(f"  TP: {buy_metrics_optuna['tp']}, FP: {buy_metrics_optuna['fp']}")
print(f"\nBest Parameters:")
for k, v in buy_params.items():
    print(f"  {k}: {v}")

Loaded Optuna results from Trial 10:
  Precision: 0.3835
  TP: 51, FP: 82

Best Parameters:
  seq_len: 12
  hidden_size: 112
  num_layers: 1
  dropout: 0.2
  lr: 0.003283136695374537
  conf_threshold: 0.7


## 2. Load Data + Model

In [3]:
def load_npz(filepath, seq_len):
    """Load NPZ Рё СЂР°Р·РґРµР»РёС‚СЊ РЅР° train/test."""
    data = np.load(filepath, allow_pickle=True)
    X = data['X'].astype(np.float32)
    y = data['y'].astype(np.int64)
    n_total = len(X)
    actual_seq = min(seq_len, N_BARS)
    X = X[:, :actual_seq, :]
    n_train = int(n_total * 0.8)
    X_train, X_test = X[:n_train], X[n_train:]
    y_train, y_test = y[:n_train], y[n_train:]
    num_classes = max(2, len(np.unique(y)))
    y_train_oh = np.eye(num_classes)[y_train]
    y_test_oh  = np.eye(num_classes)[y_test]
    print(f'  rows={n_total}, train={n_train}, test={n_total-n_train}, X={X.shape}')
    print(f'  classes: {dict(zip(*np.unique(y, return_counts=True)))}')
    train_ds = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train_oh))
    test_ds  = TensorDataset(torch.FloatTensor(X_test),  torch.FloatTensor(y_test_oh))
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)
    return train_loader, test_loader, actual_seq

# Р—Р°РіСЂСѓР¶Р°РµРј РґР°РЅРЅС‹Рµ
buy_npz = sorted(glob.glob(str(ARTIFACTS_DIR / 'data_BUY_*_normalized.npz')))
if not buy_npz:
    raise FileNotFoundError(f'No BUY NPZ in {ARTIFACTS_DIR}/')

train_loader, test_loader, seq_len_actual = load_npz(buy_npz[0], buy_params['seq_len'])
print(f"\nSeq len: {seq_len_actual}")

  rows=4156, train=3324, test=832, X=(4156, 12, 12)
  classes: {np.int64(0): np.int64(2586), np.int64(1): np.int64(1570)}

Seq len: 12


In [4]:
class GRUClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True,
                          dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size, device=x.device)
        out, _ = self.gru(x, h0)
        out = out[:, -1, :]
        out = self.dropout(out)
        return self.fc(out)

# РЎРѕР·РґР°РµРј РјРѕРґРµР»СЊ СЃ best РїР°СЂР°РјРµС‚СЂР°РјРё
model = GRUClassifier(
    N_FEATURES,
    buy_params['hidden_size'],
    buy_params['num_layers'],
    N_CLASSES,
    buy_params['dropout']
)
print(f"Model created: {model}")

Model created: GRUClassifier(
  (gru): GRU(12, 112, batch_first=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=112, out_features=2, bias=True)
)


## 3. Final Training

In [5]:
def train_final(model, train_loader, epochs, lr, device, verbose=True):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr * 0.1)
    best_acc = 0.0
    best_state = None

    it = tqdm(range(epochs), desc='Final Train', leave=True) if verbose else range(epochs)
    for epoch in it:
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch.argmax(dim=1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * X_batch.size(0)
            correct   += (y_pred.argmax(dim=1) == y_batch.argmax(dim=1)).sum().item()
            total     += X_batch.size(0)
        train_loss = total_loss / total
        train_acc  = correct / total
        scheduler.step()
        if train_acc > best_acc:
            best_acc = train_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if verbose and hasattr(it, 'set_postfix'):
            it.set_postfix({'loss': f'{train_loss:.4f}', 'acc': f'{train_acc:.4f}'})
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_acc, train_loss

# Р¤РёРЅР°Р»СЊРЅРѕРµ РѕР±СѓС‡РµРЅРёРµ
print(f"\nFinal Training with {FINAL_EPOCHS} epochs")
print(f"  Learning rate: {buy_params['lr']}")
print(f"  Dropout: {buy_params['dropout']}")
print()
model, train_acc, train_loss = train_final(model, train_loader, FINAL_EPOCHS, buy_params['lr'], DEVICE, verbose=True)
print(f"\nFinal train accuracy: {train_acc:.4f}, loss: {train_loss:.4f}")


Final Training with 100 epochs
  Learning rate: 0.003283136695374537
  Dropout: 0.2



Final Train:   0%|          | 0/100 [00:00<?, ?it/s]


Final train accuracy: 0.9723, loss: 0.0561


## 4. Export ONNX + RNN_Scaler.mqh

In [6]:
def export_to_onnx(model, seq_len, n_features, out_path, name):
    """Export to self-contained ONNX."""
    model.eval()
    model.to('cpu')
    dummy = torch.randn(1, seq_len, n_features)
    tmp = str(out_path) + '.tmp'
    old = sys.stdout
    sys.stdout = io.StringIO()
    try:
        torch.onnx.export(model, dummy, tmp,
                          input_names=['input'], output_names=['output'],
                          dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
                          opset_version=18)
    finally:
        sys.stdout = old
    m = onnx.load(tmp)
    raw_bytes = m.SerializeToString()
    if b'.onnx.data' in raw_bytes:
        raise RuntimeError(f'{name}: external .data reference!')
    with open(out_path, 'wb') as f:
        f.write(raw_bytes)
    for x in (tmp, tmp + '.data', str(out_path) + '.data'):
        if os.path.exists(x): os.remove(x)
    kb = os.path.getsize(str(out_path)) / 1024
    print(f'  {name}: {out_path} ({kb:.1f} KB)')

def generate_scaler_mqh(seq_len, n_features, eps, out_path):
    lines = [
        '//+------------------------------------------------------------------+',
        '//|                                              RNN_Scaler.mqh      |',
        '//|   Pre-normalized NPZ input (RNN_gold).                         |',
        '//|   Generated from Optuna best params: Final Training.           |',
        '//+------------------------------------------------------------------+',
        '#ifndef RNN_SCALER_MQH',
        '#define RNN_SCALER_MQH',
        '',
        f'#define NN_SEQ_LEN      {seq_len}       // BUY input window',
        f'#define NN_FEATURES      {n_features}    // features per bar',
        f'#define NN_NORM_STD_EPS  {eps:g}         // epsilon',
        '',
        '#endif // RNN_SCALER_MQH',
        '',
    ]
    with open(out_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f'  RNN_Scaler.mqh -> {out_path}')

# Р­РєСЃРїРѕСЂС‚РёСЂСѓРµРј
print(f"\n{'='*70}")
print(f"Exporting ONNX and RNN_Scaler.mqh")
print(f"{'='*70}\n")

onnx_dir = OUTPUT_DIR / 'onnx'
onnx_dir.mkdir(exist_ok=True)
export_to_onnx(model, seq_len_actual, N_FEATURES, onnx_dir / 'buy_model.onnx', 'buy')
generate_scaler_mqh(seq_len_actual, N_FEATURES, 1e-6, N_BARS, OUTPUT_DIR / 'RNN_Scaler.mqh')

# РЎРѕС…СЂР°РЅСЏРµРј РјРѕРґРµР»СЊ
models_dir = OUTPUT_DIR / 'models'
models_dir.mkdir(exist_ok=True)
torch.save(model.state_dict(), models_dir / 'model_buy_final.pth')
print(f'  Models: {models_dir}/')


Exporting ONNX and RNN_Scaler.mqh

  buy: /content/drive/MyDrive/RNN/output/onnx/buy_model.onnx (187.6 KB)
  RNN_Scaler.mqh -> /content/drive/MyDrive/RNN/output/RNN_Scaler.mqh
  Models: /content/drive/MyDrive/RNN/output/models/


## 5. Save Final Results

In [7]:
# РЎРѕС…СЂР°РЅСЏРµРј С„РёРЅР°Р»СЊРЅС‹Рµ СЂРµР·СѓР»СЊС‚Р°С‚С‹
final_summary = {
    'timestamp': datetime.now().isoformat(),
    'optuna_trial_number': best_trial_num,
    'optuna_precision': optuna_results['best_value'],
    'optuna_metrics': buy_metrics_optuna,
    'best_params': buy_params,
    'final_training': {
        'epochs': FINAL_EPOCHS,
        'train_acc': float(train_acc),
        'train_loss': float(train_loss)
    },
    'model_config': {
        'n_features': N_FEATURES,
        'n_bars': N_BARS,
        'seq_len': seq_len_actual,
    }
}

with open(OUTPUT_DIR / 'final_results.json', 'w') as f:
    json.dump(final_summary, f, indent=2)

print(f"\n{'='*70}")
print(f"FINAL TRAINING COMPLETE")
print(f"{'='*70}")
print(f"\nFinal Results:")
print(f"  Train Accuracy: {train_acc:.4f}")
print(f"  Train Loss: {train_loss:.4f}")
print(f"\nOptuna Baseline (Trial {best_trial_num}):")
print(f"  Precision: {optuna_results['best_value']:.4f}")
print(f"  TP: {buy_metrics_optuna['tp']}, FP: {buy_metrics_optuna['fp']}")
print(f"\nFiles for MT5:")
print(f"  1. {onnx_dir / 'buy_model.onnx'} -> Common\\Files\\RNN_GOLD\\buy_model.onnx")
print(f"  2. {OUTPUT_DIR / 'RNN_Scaler.mqh'} -> MQL5\\Experts\\RNN_gold_M5_BUY\\RNN_Scaler.mqh")
print(f"\nRun python training/deploy_rnn_model.py to validate and copy files to MT5.")


FINAL TRAINING COMPLETE

Final Results:
  Train Accuracy: 0.9723
  Train Loss: 0.0561

Optuna Baseline (Trial 10):
  Precision: 0.3835
  TP: 51, FP: 82

Files for MT5:
  1. /content/drive/MyDrive/RNN/output/onnx/buy_model.onnx -> Common\Files\RNN_GOLD\buy_model.onnx
  2. /content/drive/MyDrive/RNN/output/RNN_Scaler.mqh -> MQL5\Experts\RNN_gold_M5_BUY\RNN_Scaler.mqh

Run python training/deploy_rnn_model.py to validate and copy files to MT5.
